In [1]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium shimmy optuna higher

In [2]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import mplfinance as mpf
import pandas_ta as ta
import pygame
import os
import pytz
import json
import re
import random
import pickle

import gymnasium as gym
from gymnasium import spaces
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Dropout, Attention, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.callbacks import Callback, ModelCheckpoint
from scipy.signal import argrelextrema
import tensorflow as tf
from collections import deque
from tensorflow import keras
from tensorflow.keras import layers, optimizers, models
from scipy.signal import find_peaks

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
def get_index_symbol_and_quantity(index):

    # Dictionary mapping index name to index symbols
    index_symbols = {
        'Bankex': 'BSE:BANKEX-INDEX',
        'Finnifty': 'NSE:FINNIFTY-INDEX',
        'Bank Nifty': 'NSE:NIFTYBANK-INDEX',
        'Nifty': 'NSE:NIFTY50-INDEX',
        'Sensex': 'BSE:SENSEX-INDEX'
    }

    # Determine the index symbol for the given index
    index_symbol = index_symbols.get(index, 'Invalid Index')

    # Determine the quantity based on the index symbol
    if index_symbol == "NSE:NIFTY50-INDEX":
        quantity = 25  # 25 is one lot for Nifty
    elif index_symbol == "NSE:NIFTYBANK-INDEX":
        quantity = 15  # 15 is one lot for Bank Nifty
    elif index_symbol == "NSE:FINNIFTY-INDEX":
        quantity = 40  # 40 is one lot for Finnifty
    elif index_symbol == "BSE:SENSEX-INDEX":
        quantity = 20  # 20 is two lot for Sensex
    elif index_symbol == "BSE:BANKEX-INDEX":
        quantity = 15  # 15 is one lot for Bankex
    else:
        quantity = 0  # Default value if none of the conditions match

    return index_symbol, quantity

In [4]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

index_symbol, quantity = get_index_symbol_and_quantity("Bank Nifty")

interval_minutes = 2 # Set the interval to 1, 5, or 15 minutes

ist_timezone = pytz.timezone("Asia/Kolkata")

# Initialize the mixer
#pygame.mixer.init()

# Load the sound file
#pygame.mixer.music.load('sounds/alert.mp3')

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

target = 80
trailing_sl = 40

brokerage = 100

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [5]:
 session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [6]:
def fetch_candle_data(number):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                train_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                return train_df
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [7]:
def fetch_train_candle_data(days_count):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [8]:
#train_df = fetch_candle_data(100)

train_df = fetch_train_candle_data(25)

print(len(train_df))

train_df = train_df.drop_duplicates(subset='datetime', keep='first')

print(len(train_df))

320453
317445


In [9]:
class FullFeaturePipeline:
    def __init__(self, df: pd.DataFrame, base_interval_minutes: int):
        """
        df: DataFrame with columns [datetime, open, high, low, close, volume (optional)]
            'datetime' is a Unix timestamp in seconds.
        base_interval_minutes: The base timeframe in minutes (e.g. 2, 5, 15, etc.).
        """
        self.df = df.copy()
        self.base_interval = base_interval_minutes

        # We no longer need an htf_map since higher timeframe features are removed
        # self.htf_map = { ... }  # Removed

    def preprocess_datetime(self):
        """
        1) Convert Unix timestamp to local time (Asia/Kolkata).
        2) Check for duplicates/missing timestamps.
        3) Sort by time and set 'datetime' as the index.
        """
        ist = timezone('Asia/Kolkata')

        self.df['datetime'] = pd.to_datetime(self.df['datetime'], unit='s')
        self.df['datetime'] = (self.df['datetime']
                               .dt.tz_localize('UTC')
                               .dt.tz_convert(ist)
                               .dt.tz_localize(None))

        # Check for duplicates/missing
        if self.df['datetime'].duplicated().any() or self.df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        # Sort and set index
        self.df.sort_values('datetime', inplace=True)
        self.df.set_index('datetime', inplace=True)
        return self

    def clean_data(self):
        """
        Optionally drop 'volume' column if it contains zeros or NaNs.
        Adjust if you prefer different volume handling.
        """
        if 'volume' in self.df.columns:
            if self.df['volume'].isnull().any() or (self.df['volume'] == 0).any():
                self.df.drop('volume', axis=1, inplace=True, errors='ignore')
        return self

    def add_base_timeframe_features(self):
        """
        Compute a few standard technical indicators on the base timeframe:
        RSI, MACD, Bollinger Bands, ATR, etc.
        """
        self.df.sort_index(inplace=True)

        # RSI on base timeframe close
        self.df['rsi_base'] = ta.rsi(self.df['close'], length=14)

        # MACD
        macd = ta.macd(self.df['close'], fast=12, slow=26, signal=9)
        self.df['macd'] = macd['MACD_12_26_9']
        self.df['macd_h'] = macd['MACDh_12_26_9']
        self.df['macd_s'] = macd['MACDs_12_26_9']

        # Bollinger Bands
        bbands = ta.bbands(self.df['close'], length=20, std=2)
        self.df['bb_lower'] = bbands['BBL_20_2.0']
        self.df['bb_mid'] = bbands['BBM_20_2.0']
        self.df['bb_upper'] = bbands['BBU_20_2.0']

        # ATR (for baseline volatility measure)
        self.df['atr_base'] = ta.atr(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        )

        return self

    def add_candlestick_patterns(self):
        """
        Detect a few common candlestick patterns.
        We store them as numeric columns (0 = not present, 1 = bullish pattern, -1 = bearish pattern, etc.)
        so that the pipeline remains consistent with numeric-only scaling.
        """
        self.df.sort_index(inplace=True)

        # Example: Doji detection (very simplistic)
        # We check if the open and close are nearly the same
        body_size = (self.df['close'] - self.df['open']).abs()
        candle_range = self.df['high'] - self.df['low']
        self.df['candle_doji'] = np.where(
            (body_size <= 0.1 * candle_range), 1.0, 0.0
        )

        # Example: Engulfing pattern (bullish and bearish)
        # - Bullish Engulfing: today's open < yesterday's close, today's close > yesterday's open
        # - Bearish Engulfing: today's open > yesterday's close, today's close < yesterday's open
        # We shift the open/close by 1 day to compare
        self.df['open_shift1'] = self.df['open'].shift(1)
        self.df['close_shift1'] = self.df['close'].shift(1)

        cond_bull_engulf = (self.df['open'] < self.df['close_shift1']) & (self.df['close'] > self.df['open_shift1'])
        cond_bear_engulf = (self.df['open'] > self.df['close_shift1']) & (self.df['close'] < self.df['open_shift1'])

        self.df['candle_engulf'] = np.where(cond_bull_engulf, 1.0,
                                    np.where(cond_bear_engulf, -1.0, 0.0))

        # Drop helper columns to keep dataset clean
        self.df.drop(['open_shift1', 'close_shift1'], axis=1, inplace=True)

        return self

    def add_swing_points(self, left_bars=3, right_bars=3):
        """
        Detect local swing highs and lows.
        left_bars/right_bars define how many bars on each side must be lower/higher (for highs/lows).
        """
        self.df.sort_index(inplace=True)

        # For each row, check if 'high' is greater than the preceding and following N bars
        rolling_high = self.df['high'].rolling(left_bars+1, center=False).max()
        rolling_low = self.df['low'].rolling(left_bars+1, center=False).min()

        # We'll do a simplistic approach: shift rolling computations accordingly
        # A more robust approach might be to iterate row by row, but we'll keep it vectorized for brevity.

        # Swing High
        # Compare current high to future bars as well, so we do a forward rolling
        forward_high = self.df['high'].shift(-right_bars).rolling(right_bars+1, center=False).max()
        self.df['is_swing_high'] = 0.0
        cond_high = (self.df['high'] == rolling_high) & (self.df['high'] == forward_high)
        self.df.loc[cond_high, 'is_swing_high'] = 1.0

        # Swing Low
        forward_low = self.df['low'].shift(-right_bars).rolling(right_bars+1, center=False).min()
        self.df['is_swing_low'] = 0.0
        cond_low = (self.df['low'] == rolling_low) & (self.df['low'] == forward_low)
        self.df.loc[cond_low, 'is_swing_low'] = 1.0

        return self

    def add_range_breakout_features(self, window=20):
        """
        Add Donchian channels, breakout flags, and a simple measure of range expansion.
        """
        self.df.sort_index(inplace=True)

        # Donchian Channels
        self.df['donchian_high'] = self.df['high'].rolling(window).max()
        self.df['donchian_low'] = self.df['low'].rolling(window).min()
        self.df['donchian_range'] = self.df['donchian_high'] - self.df['donchian_low']

        # Breakout flags
        #  1.0 if close > donchian_high, -1.0 if close < donchian_low, else 0.0
        self.df['donchian_breakout'] = np.where(
            self.df['close'] > self.df['donchian_high'], 1.0,
            np.where(self.df['close'] < self.df['donchian_low'], -1.0, 0.0)
        )

        # Range expansion/compression example
        # Rolling std of (high - low)
        self.df['range_expansion'] = (self.df['high'] - self.df['low']).rolling(window).std()

        return self

    def add_momentum_features(self):
        """
        Add additional momentum-based indicators like Stochastics, ADX, CCI, etc.
        (We already have RSI, MACD in add_base_timeframe_features, but you can unify them here if you prefer.)
        """
        self.df.sort_index(inplace=True)

        # Stochastic
        stoch = ta.stoch(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            k=14, d=3
        )
        self.df['stoch_k'] = stoch['STOCHk_14_3_3']
        self.df['stoch_d'] = stoch['STOCHd_14_3_3']

        # ADX (Average Directional Movement Index)
        adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
        self.df['adx'] = adx_data['ADX_14']
        self.df['diplus'] = adx_data['DMP_14']
        self.df['diminus'] = adx_data['DMN_14']

        # CCI (Commodity Channel Index)
        self.df['cci'] = ta.cci(self.df['high'], self.df['low'], self.df['close'], length=20)

        return self

    def add_volatility_features(self, window=20):
        """
        Add additional volatility-based features, e.g. historical volatility, Z-score of returns, etc.
        """
        self.df.sort_index(inplace=True)

        # Historical volatility (using close-to-close log returns)
        self.df['log_return'] = np.log(self.df['close'] / self.df['close'].shift(1))
        self.df['hist_vol'] = self.df['log_return'].rolling(window).std() * np.sqrt(252)  # Annualized approximation

        # Z-Score of Price Changes over 'window'
        rolling_mean = self.df['log_return'].rolling(window).mean()
        rolling_std = self.df['log_return'].rolling(window).std()
        self.df['zscore_return'] = (self.df['log_return'] - rolling_mean) / (rolling_std + 1e-9)

        return self

    def add_volume_features(self):
        """
        Add volume-based features if 'volume' is present.
        """
        if 'volume' not in self.df.columns:
            return self  # No volume data, skip

        self.df.sort_index(inplace=True)

        # On-Balance Volume
        self.df['obv'] = ta.obv(self.df['close'], self.df['volume'])

        # Volume spike detection:
        # Compare current volume to rolling average
        vol_mean = self.df['volume'].rolling(20).mean()
        vol_std = self.df['volume'].rolling(20).std()
        self.df['vol_spike'] = np.where(
            (self.df['volume'] > (vol_mean + 2 * vol_std)), 1.0, 0.0
        )

        # VWAP over the day (if you have daily or session logic, you'd groupby date)
        # Here, for a simpler approach, we do a rolling 1-day approximation:
        # This might not be perfectly accurate if you have continuous 24h data, but it’s a start.
        # Usually you’d reset VWAP each day or each session.
        typical_price = (self.df['high'] + self.df['low'] + self.df['close']) / 3.0
        self.df['cum_tp_vol'] = (typical_price * self.df['volume']).cumsum()
        self.df['cum_vol'] = self.df['volume'].cumsum()
        self.df['vwap'] = self.df['cum_tp_vol'] / (self.df['cum_vol'] + 1e-9)

        # Drop helper columns
        self.df.drop(['cum_tp_vol', 'cum_vol'], axis=1, inplace=True)

        return self

    def add_regime_features(self):
        """
        Classify each bar as 'trending' or 'ranging' (0 or 1) based on ADX, or volatility-based thresholds, etc.
        Keep it numeric for scaling (e.g., 0.0 or 1.0).
        """
        self.df.sort_index(inplace=True)

        # If ADX above 25 => trending, else ranging
        # (25 is just a typical reference threshold; pick whichever fits your strategy)
        if 'adx' not in self.df.columns:
            # If ADX wasn't yet computed, do it quickly
            adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
            self.df['adx'] = adx_data['ADX_14']

        self.df['regime_trend'] = np.where(self.df['adx'] >= 25, 1.0, 0.0)

        # Alternatively, you could have multiple columns for different regime classifications
        return self

    def add_time_features(self):
        """
        Extract cyclical or numeric time features (hour of day, day of week).
        We ensure they are numeric (float).
        """
        self.df.sort_index(inplace=True)

        # Because 'datetime' is now our index, we can access it via self.df.index
        # Create columns for hour, day of week
        self.df['hour'] = self.df.index.hour.astype(float)
        self.df['day_of_week'] = self.df.index.dayofweek.astype(float)

        # Cyclical encoding for hour
        self.df['hour_sin'] = np.sin(2 * np.pi * self.df['hour'] / 24.0)
        self.df['hour_cos'] = np.cos(2 * np.pi * self.df['hour'] / 24.0)

        # Cyclical encoding for day_of_week
        self.df['dow_sin'] = np.sin(2 * np.pi * self.df['day_of_week'] / 7.0)
        self.df['dow_cos'] = np.cos(2 * np.pi * self.df['day_of_week'] / 7.0)

        # You can drop raw hour/day_of_week if you prefer only cyclical features
        # self.df.drop(['hour', 'day_of_week'], axis=1, inplace=True)

        return self

    def add_adaptive_targets_and_stops(self):
        """
        Instead of fixed (2 * ATR) / (1 * ATR), adapt based on current volatility or regime.
        For demonstration, we’ll say:
            - If regime is trending, target = 3 * ATR, stop = 1.5 * ATR
            - If regime is ranging, target = 1.5 * ATR, stop = 1 * ATR
        This is just an example logic.
        """
        self.df.sort_index(inplace=True)

        if 'atr_base' not in self.df.columns:
            self.df['atr_base'] = ta.atr(self.df['high'], self.df['low'], self.df['close'], length=14)

        # If we haven't computed 'regime_trend', do so
        if 'regime_trend' not in self.df.columns:
            self.add_regime_features()

        # Example adaptation
        is_trend = (self.df['regime_trend'] == 1.0)
        self.df['Target'] = np.where(is_trend, 3.0 * self.df['atr_base'], 1.5 * self.df['atr_base'])
        self.df['StopLoss'] = np.where(is_trend, 1.5 * self.df['atr_base'], 1.0 * self.df['atr_base'])

        return self

    def run_pipeline(self):
        """
        Run all steps except higher timeframe computations (now removed).
        You can rearrange the order as desired.
        """
        (self.preprocess_datetime()
             .clean_data()
             .add_base_timeframe_features()
             .add_candlestick_patterns()
             .add_swing_points()
             .add_range_breakout_features()
             .add_momentum_features()
             .add_volatility_features()
             .add_volume_features()            # only applies if 'volume' exists
             .add_regime_features()
             .add_time_features()
             .add_adaptive_targets_and_stops()
        )
        return self

    def get_processed_df(self):
        """
        Retrieve the final DataFrame (including any signals if label_signals() was called).
        We'll also ensure that all columns except 'Signal' are float with 2 decimal places.
        """
        # Drop rows that have NaN from lookbacks
        self.df.dropna(axis=0, how='any', inplace=True)

        # Drop 'Entry Price' and 'Exit Price' if you do not want them in final features
        if 'Entry Price' in self.df.columns:
            self.df.drop('Entry Price', axis=1, inplace=True)
        if 'Exit Price' in self.df.columns:
            self.df.drop('Exit Price', axis=1, inplace=True)

        # Ensure 'Signal' is int; everything else float with 2 decimals
        if 'Signal' in self.df.columns:
            sig_col = self.df['Signal'].astype(int)
            self.df.drop(columns=['Signal'], inplace=True)
        else:
            sig_col = None

        # Convert all remaining columns to float
        for col in self.df.columns:
            self.df[col] = self.df[col].astype(float)

        # Round to 2 decimals
        self.df = self.df.round(2)

        # Reinsert Signal column if it exists
        if sig_col is not None:
            self.df['Signal'] = sig_col

        return self.df

train_pipeline = FullFeaturePipeline(train_df, interval_minutes)
train_pipeline.run_pipeline()

processed_train_df = train_pipeline.get_processed_df()
processed_train_df

,open,high,low,close,rsi_base,macd,macd_h,macd_s,bb_lower,bb_mid,...,zscore_return,regime_trend,hour,day_of_week,hour_sin,hour_cos,dow_sin,dow_cos,Target,StopLoss
datetime,,,,,,,,,,,,,,,,,,,,,
2017-12-11 10:21:00,25405.20,25405.20,25395.50,25403.30,33.35,-3.38,-3.59,0.21,25401.65,25420.66,...,0.06,0.0,10.0,0.0,0.50,-0.87,0.00,1.00,17.79,11.86
2017-12-11 10:23:00,25402.10,25418.60,25402.00,25417.20,49.12,-2.85,-2.45,-0.40,25401.31,25420.25,...,2.28,0.0,10.0,0.0,0.50,-0.87,0.00,1.00,18.34,12.23
2017-12-11 10:25:00,25416.50,25425.20,25414.20,25415.70,47.80,-2.52,-1.69,-0.82,25400.88,25419.78,...,-0.16,0.0,10.0,0.0,0.50,-0.87,0.00,1.00,18.20,12.13
2017-12-11 10:27:00,25415.90,25421.30,25411.10,25415.30,47.44,-2.26,-1.15,-1.11,25400.60,25419.60,...,-0.04,0.0,10.0,0.0,0.50,-0.87,0.00,1.00,17.98,11.99
2017-12-11 10:29:00,25415.90,25418.20,25408.30,25415.10,47.24,-2.05,-0.75,-1.30,25400.23,25419.32,...,0.01,0.0,10.0,0.0,0.50,-0.87,0.00,1.00,17.74,11.83
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-15 15:21:00,51907.25,51936.00,51905.85,51936.00,59.03,11.11,6.96,4.14,51824.41,51883.28,...,0.85,0.0,15.0,1.0,-0.71,-0.71,0.78,0.62,57.43,38.29
2024-10-15 15:23:00,51935.85,51955.75,51932.80,51950.15,61.10,14.32,8.14,6.18,51826.39,51888.73,...,0.32,0.0,15.0,1.0,-0.71,-0.71,0.78,0.62,55.78,37.19
2024-10-15 15:25:00,51950.80,51984.15,51950.10,51978.35,64.92,18.92,10.19,8.73,51820.56,51893.80,...,0.86,0.0,15.0,1.0,-0.71,-0.71,0.78,0.62,55.45,36.97


In [10]:
train_pipeline.get_processed_df().columns

Index(['open', 'high', 'low', 'close', 'rsi_base', 'macd', 'macd_h', 'macd_s',
       'bb_lower', 'bb_mid', 'bb_upper', 'atr_base', 'candle_doji',
       'candle_engulf', 'is_swing_high', 'is_swing_low', 'donchian_high',
       'donchian_low', 'donchian_range', 'donchian_breakout',
       'range_expansion', 'stoch_k', 'stoch_d', 'adx', 'diplus', 'diminus',
       'cci', 'log_return', 'hist_vol', 'zscore_return', 'regime_trend',
       'hour', 'day_of_week', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
       'Target', 'StopLoss'],
      dtype='object')

MAML RL Model

In [11]:
#!/usr/bin/env python
# coding: utf-8

"""
Answering two main questions in a single Jupyter notebook:
1) Whether shuffling is good for time-series trading data.
2) What num_tasks, task_size, etc. do in the context of MAML training.

Because we’re operating in a time-series environment, data order matters
(timestamps), so typically we avoid random shuffling. MAML can still use
different segments of the time series as different tasks without losing
temporal structure.

Usage instructions:
1. Copy or download this notebook.
2. Run all cells in order.
3. The final cell prints the explanation.
"""

import sys

def explanation():
    """
    Returns a string explaining:
      1. Why we avoid shuffling in time-series trading data.
      2. The meaning of num_tasks, task_size, etc. in the MAML approach.
    """
    msg = """
1) Shuffling Time-Series Trading Data
-------------------------------------
Shuffling is generally NOT recommended for time-series trading data, because time-ordering is crucial to avoid forward-looking bias (data leakage).
- If you randomly shuffle rows, your model might see future prices before earlier ones, destroying the causal relationships.
- Therefore, we usually keep the data in chronological order.
- For meta-learning (like MAML), we can still 'segment' the time series into multiple tasks, but within each segment we maintain chronological order.
  That way each task is a contiguous chunk of time, preserving proper temporal relationships.

2) num_tasks, task_size, etc.
-----------------------------
- 'num_tasks' and 'task_size' refer to how we slice or sample from our dataset for the meta-learning approach.
- 'task_size' is how many data points (time steps) each environment (or 'task') will contain.
  For example, if task_size=1000, each chunk/environment covers 1000 sequential bars/candles.
- 'num_tasks' is how many separate tasks you want to sample or create from the dataset.
  For instance, if 'num_tasks'=5, you might create 5 different slices (or more if you shuffle them) of the data, each with 'task_size' bars.
- Each task (slice) is treated as a separate environment for MAML. The meta-learner tries to learn a policy that can quickly adapt to each slice's unique behaviors.
- 'shuffle_tasks' determines whether these slices are chosen randomly or in contiguous order. Even if you shuffle tasks at the slice level,
  each slice itself should remain time-sequential internally.
- This approach is intended to expose the meta-learning algorithm to various regimes or patterns, so the final meta-policy can adapt quickly to new data segments.

Summary
-------
- Do NOT shuffle individual rows for time-series.
- Use 'num_tasks' and 'task_size' to define how many separate environment slices you want for meta-learning.
- This preserves chronological order within each slice but still allows variety across tasks.
"""
    return msg

In [12]:
#!/usr/bin/env python
# coding: utf-8

"""
MAML + Transformer for Discrete Action Trading with Stop/Target Logic
and Scale-Invariant Rewards (Percentage-Based PnL). This version is
reworked to ensure the reward/penalty mechanism is fully neutral to
different instruments (stocks, indexes, crypto, etc.) by:
  1) Removing any need for 'StopLoss' or 'Target' columns in the data,
     instead using fixed percentage-based stop and target from config.
  2) Keeping the reward purely in percentage terms, i.e. scale-invariant.
  3) Keeping the penalty terms (holding_penalty, repeat_action_penalty)
     as constants that are also independent of the price scale.

Required Python libraries:
    - numpy
    - pandas
    - torch
    - higher
    - matplotlib
"""

################################################################################
# 0. Imports and Setup
################################################################################

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import random
import os
import higher
from typing import List, Tuple, Optional

%matplotlib inline

# Detect device (CPU/GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

################################################################################
# 1. Configuration Dictionary
################################################################################

config = {
    # Environment hyperparams
    "seq_len": 16,                  # Number of timesteps (bars) per observation
    "initial_capital": 1_000_000.0, # Starting capital (for final PnL calculations)
    "quantity": 50,                 # Position size (if needed; environment is % based)
    "holding_penalty": 0.01,        # Penalty per bar while in a position
    "repeat_action_penalty": 0.01,  # Penalty if the same action is repeated 4+ times
    "reward_scale": 1.0,            # Factor to scale PnL reward (already % based)
    "max_steps_per_env": 200,       # Maximum steps in an episode
    "max_position_bars": 9999,      # Force-close after this many bars if still in position
    "max_bars_after_target": 30,    # Force-close 30 bars after the target is first crossed

    # **New** scale-invariant parameters (no reliance on DataFrame columns)
    # For any new trade:
    #   Long: stop_loss = entry_price * (1 - stop_pct)
    #         target_price = entry_price * (1 + target_pct)
    #   Short: stop_loss = entry_price * (1 + stop_pct)
    #          target_price = entry_price * (1 - target_pct)
    "stop_pct": 0.01,      # e.g. 1% stop
    "target_pct": 0.02,    # e.g. 2% target

    # MAML hyperparams
    "lr_inner": 1e-3,       # Learning rate for inner-loop adaptation
    "lr_meta": 1e-4,        # Learning rate for meta-optimizer
    "gamma": 0.99,          # Discount factor for computing returns
    "inner_steps": 2,       # Number of gradient steps per inner adaptation
    "weight_decay": 1e-5,   # Weight decay for Adam
    "grad_clip": 5.0,       # Gradient clipping

    # Training tasks setup
    "num_tasks": 3,         # Number of separate tasks (data slices)
    "task_size": 1000,      # Length of each slice
    "num_meta_iterations": 50,
    "shuffle_tasks": True,

    # Model file
    "model_file": "maml_trader_policy.pt",

    # Exploration temperature (larger = more random, smaller = greedier)
    "exploration_temperature": 1.0
}

################################################################################
# 2. Environment with Scale-Invariant Stop/Target Logic + Forced Close
################################################################################

class StopTargetTradingEnv:
    """
    Discrete-action environment for trading with purely percentage-based
    StopLoss/Target logic, plus a forced close if holding beyond a certain
    threshold. Rewards are scale-invariant (percentage-based PnL).

    Actions (integer):
      0 = HOLD
      1 = BUY
      2 = SELL
      3 = CLOSE

    Required column in df: 'close' (the environment uses only 'close' here
    to measure current_price). No 'StopLoss' or 'Target' columns needed.

    For any new position:
      - We compute stop_loss and target_price as scale-invariant:
          Long:  stop_loss   = entry_price * (1 - stop_pct)
                 target_price = entry_price * (1 + target_pct)
          Short: stop_loss   = entry_price * (1 + stop_pct)
                 target_price = entry_price * (1 - target_pct)

      - On each step:
         * If price crosses stop, forced close with PnL.
         * If price crosses target, we ALLOW the agent to close with action=3.
           But if the agent does not close, after `max_bars_after_target` bars
           we forcibly close at the market price.
      - We also forcibly close if the position is open for `max_position_bars`
        bars or if we reach the end of the data.

    Reward each step:
       - If we close a position, reward = %PNL * reward_scale
       - If still in a position, we subtract a small holding_penalty
       - If the same action is repeated 4+ times in a row, we subtract
         repeat_action_penalty

    Because all PnL is computed as (exit_price - entry_price)/entry_price
    (long) or (entry_price - exit_price)/entry_price (short), the environment
    is neutral to absolute price levels across different assets.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        seq_len: int,
        quantity: int,
        holding_penalty: float,
        repeat_action_penalty: float,
        reward_scale: float,
        max_position_bars: int,
        max_bars_after_target: int,
        stop_pct: float,
        target_pct: float
    ):
        # Basic checks
        self.df = df.copy()
        self.df.reset_index(drop=True, inplace=True)

        if 'close' not in self.df.columns:
            raise ValueError("DataFrame must contain 'close' column for current price data.")

        self.seq_len = seq_len
        self.quantity = quantity
        self.holding_penalty = holding_penalty
        self.repeat_action_penalty = repeat_action_penalty
        self.reward_scale = reward_scale
        self.max_position_bars = max_position_bars
        self.max_bars_after_target = max_bars_after_target
        self.stop_pct = stop_pct
        self.target_pct = target_pct

        self.current_step = 0
        self.done = False

        # Position tracking
        self.position = None
        self.entry_price = 0.0
        self.stop_loss = 0.0
        self.target_price = 0.0
        self.time_in_position = 0
        self.target_crossed = False
        self.bars_after_target = 0

        # Action repetition
        self.last_action = None
        self.repeat_count = 0

        # Logging
        self.trade_log = []
        self.max_step = len(self.df) - 1

    def _get_observation(self) -> np.ndarray:
        """
        Return the last seq_len rows as a (seq_len, num_features) array.
        If not enough rows, pad with zeros at the front.
        """
        start_idx = max(0, self.current_step - self.seq_len + 1)
        obs_data = self.df.iloc[start_idx : self.current_step + 1].values
        if len(obs_data) < self.seq_len:
            pad_len = self.seq_len - len(obs_data)
            obs_data = np.concatenate(
                [np.zeros((pad_len, obs_data.shape[1])), obs_data],
                axis=0
            )
        return obs_data

    def reset(self):
        self.current_step = 0
        self.done = False

        self.position = None
        self.entry_price = 0.0
        self.stop_loss = 0.0
        self.target_price = 0.0
        self.time_in_position = 0
        self.target_crossed = False
        self.bars_after_target = 0

        self.last_action = None
        self.repeat_count = 0

        self.trade_log = []
        return self._get_observation()

    def step(self, action: int):
        """
        Execute one step in the environment with the given action.
        """
        if self.done:
            return self._get_observation(), 0.0, True, {}

        row = self.df.iloc[self.current_step]
        current_price = float(row['close'])
        step_reward = 0.0
        info = {}

        # Track repeated action
        if self.last_action == action:
            self.repeat_count += 1
        else:
            self.repeat_count = 1
        self.last_action = action

        # Check forced stop-loss logic if we have a position
        if self.position is not None:
            # For a long: forced close if current_price <= stop_loss
            # For a short: forced close if current_price >= stop_loss
            if self.position == 'long' and current_price <= self.stop_loss:
                pnl_pct = (current_price - self.entry_price) / self.entry_price
                step_reward += pnl_pct * self.reward_scale

                self.trade_log.append({
                    "Step": self.current_step,
                    "Position": self.position,
                    "EntryPrice": self.entry_price,
                    "ExitPrice": current_price,
                    "PnL%": pnl_pct * 100.0,
                    "StopHit": True,
                    "TargetHit": False
                })
                self._clear_position()

            elif self.position == 'short' and current_price >= self.stop_loss:
                pnl_pct = (self.entry_price - current_price) / self.entry_price
                step_reward += pnl_pct * self.reward_scale

                self.trade_log.append({
                    "Step": self.current_step,
                    "Position": self.position,
                    "EntryPrice": self.entry_price,
                    "ExitPrice": current_price,
                    "PnL%": pnl_pct * 100.0,
                    "StopHit": True,
                    "TargetHit": False
                })
                self._clear_position()

        # Normal action logic
        if action == 1:  # BUY
            if self.position is None:
                self.position = 'long'
                self.entry_price = current_price
                self.stop_loss = self.entry_price * (1.0 - self.stop_pct)
                self.target_price = self.entry_price * (1.0 + self.target_pct)
                self.time_in_position = 0
                self.target_crossed = False
                self.bars_after_target = 0

        elif action == 2:  # SELL
            if self.position is None:
                self.position = 'short'
                self.entry_price = current_price
                self.stop_loss = self.entry_price * (1.0 + self.stop_pct)
                self.target_price = self.entry_price * (1.0 - self.target_pct)
                self.time_in_position = 0
                self.target_crossed = False
                self.bars_after_target = 0

        elif action == 3:  # CLOSE
            # Only allow close if a position is open AND we've crossed target
            if self.position is not None and self.target_crossed:
                if self.position == 'long':
                    pnl_pct = (current_price - self.entry_price) / self.entry_price
                else:  # short
                    pnl_pct = (self.entry_price - current_price) / self.entry_price

                step_reward += pnl_pct * self.reward_scale
                self.trade_log.append({
                    "Step": self.current_step,
                    "Position": self.position,
                    "EntryPrice": self.entry_price,
                    "ExitPrice": current_price,
                    "PnL%": pnl_pct * 100.0,
                    "StopHit": False,
                    "TargetHit": True  # By definition, close only if target crossed
                })
                self._clear_position()

        # Holding logic if still in a position
        if self.position is not None:
            self.time_in_position += 1
            # small penalty each bar
            step_reward -= self.holding_penalty

            # Force close if we've been in a position too long
            if self.time_in_position >= self.max_position_bars:
                if self.position == 'long':
                    pnl_pct = (current_price - self.entry_price) / self.entry_price
                else:
                    pnl_pct = (self.entry_price - current_price) / self.entry_price

                step_reward += pnl_pct * self.reward_scale
                self.trade_log.append({
                    "Step": self.current_step,
                    "Position": self.position,
                    "EntryPrice": self.entry_price,
                    "ExitPrice": current_price,
                    "PnL%": pnl_pct * 100.0,
                    "StopHit": False,
                    "TargetHit": False,
                    "ForcedCloseMaxBars": True
                })
                self._clear_position()
            else:
                # Check if target is crossed
                if not self.target_crossed:
                    if self.position == 'long' and current_price >= self.target_price:
                        self.target_crossed = True
                        self.bars_after_target = 0
                    elif self.position == 'short' and current_price <= self.target_price:
                        self.target_crossed = True
                        self.bars_after_target = 0

                # If target was crossed, track how many bars since crossing
                if self.target_crossed:
                    self.bars_after_target += 1
                    if self.bars_after_target >= self.max_bars_after_target:
                        # Force close after target
                        if self.position == 'long':
                            pnl_pct = (current_price - self.entry_price) / self.entry_price
                        else:
                            pnl_pct = (self.entry_price - current_price) / self.entry_price

                        step_reward += pnl_pct * self.reward_scale
                        self.trade_log.append({
                            "Step": self.current_step,
                            "Position": self.position,
                            "EntryPrice": self.entry_price,
                            "ExitPrice": current_price,
                            "PnL%": pnl_pct * 100.0,
                            "StopHit": False,
                            "TargetHit": True,
                            "ForcedCloseAfterTarget": True
                        })
                        self._clear_position()

        # penalty if repeating the same action too often
        if self.repeat_count >= 4:
            step_reward -= self.repeat_action_penalty

        # increment step
        self.current_step += 1
        if self.current_step >= self.max_step:
            self.done = True
            # if position still open, forcibly close at final price
            if self.position is not None:
                final_price = float(self.df.iloc[self.current_step - 1]['close'])
                if self.position == 'long':
                    pnl_pct = (final_price - self.entry_price) / self.entry_price
                else:
                    pnl_pct = (self.entry_price - final_price) / self.entry_price
                step_reward += pnl_pct * self.reward_scale

                self.trade_log.append({
                    "Step": self.current_step,
                    "Position": self.position,
                    "EntryPrice": self.entry_price,
                    "ExitPrice": final_price,
                    "PnL%": pnl_pct * 100.0,
                    "StopHit": False,
                    "TargetHit": False,
                    "ForcedCloseEnd": True
                })
                self._clear_position()

        return self._get_observation(), step_reward, self.done, info

    def _clear_position(self):
        """
        Helper to reset position-related variables.
        """
        self.position = None
        self.entry_price = 0.0
        self.stop_loss = 0.0
        self.target_price = 0.0
        self.time_in_position = 0
        self.target_crossed = False
        self.bars_after_target = 0

    def get_trade_log(self) -> pd.DataFrame:
        return pd.DataFrame(self.trade_log)


################################################################################
# 3. Transformer Policy (batch_first=True)
################################################################################

class DeeperTransformerPolicy(nn.Module):
    """
    A multi-layer Transformer-based policy for discrete action logits:
    Actions: 0=Hold, 1=Buy, 2=Sell, 3=Close.
    """

    def __init__(
        self,
        num_features: int,
        num_actions: int = 4,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 3,
        dim_feedforward: int = 256,
        dropout: float = 0.2
    ):
        super().__init__()

        self.num_features = num_features
        self.d_model = d_model

        # Linear embedding to project input features -> d_model
        self.embedding = nn.Linear(num_features, d_model)

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='relu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # MLP head for action logits
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_actions)
        )

    def forward(self, x: torch.Tensor):
        """
        :param x: shape (batch_size, seq_len, num_features)
        :return: logits, shape (batch_size, num_actions)
        """
        x_emb = self.embedding(x)                # (batch_size, seq_len, d_model)
        x_enc = self.transformer_encoder(x_emb)  # (batch_size, seq_len, d_model)
        x_final = x_enc[:, -1, :]                # take last timestep
        logits = self.head(x_final)              # (batch_size, num_actions)
        return logits


################################################################################
# 4. MAML Trader
################################################################################

class AdvancedMAMLTrader:
    """
    MAML-style approach for discrete RL with 'higher'. It collects episodes
    by sampling actions from the current policy distribution (with optional
    temperature for exploration), then does inner and meta updates.

    NOTE: The environment is time-series-based. We never shuffle data in a
    single environment slice to avoid lookahead bias.
    """

    def __init__(self, policy: nn.Module, cfg: dict):
        self.device = device
        self.policy = policy.to(self.device)

        self.lr_inner = cfg["lr_inner"]
        self.lr_meta = cfg["lr_meta"]
        self.gamma = cfg["gamma"]
        self.inner_steps = cfg["inner_steps"]
        self.max_steps_per_env = cfg["max_steps_per_env"]
        self.weight_decay = cfg["weight_decay"]
        self.grad_clip = cfg["grad_clip"]

        # Additional param for exploration
        self.temperature = cfg.get("exploration_temperature", 1.0)

        self.meta_optimizer = optim.Adam(
            self.policy.parameters(),
            lr=self.lr_meta,
            weight_decay=self.weight_decay
        )

    def collect_episodes(
        self,
        env: StopTargetTradingEnv,
        model: nn.Module,
        max_steps=None
    ):
        """
        Roll out an episode (up to max_steps) using the given model in sampling mode,
        with a softmax policy scaled by self.temperature for exploration.
        """
        if max_steps is None:
            max_steps = self.max_steps_per_env

        transitions = []
        obs = env.reset()
        done = False
        step_count = 0

        while not done and step_count < max_steps:
            obs_np = np.array([obs], dtype=np.float32)
            obs_t = torch.from_numpy(obs_np).to(self.device)

            with torch.no_grad():
                logits = model(obs_t)
                # Clean up any NaN or Inf in logits
                logits = torch.nan_to_num(logits, nan=0.0, posinf=1e3, neginf=-1e3)

                # Scale logits by 1/temperature for exploration (softmax with temperature)
                scaled_logits = logits / max(self.temperature, 1e-8)
                probs = F.softmax(scaled_logits, dim=-1)

                # Also ensure no numerical issues in probabilities
                probs = torch.nan_to_num(probs, nan=1e-8)
                probs = probs / probs.sum(dim=-1, keepdim=True)

                action = np.random.choice(len(probs[0]), p=probs[0].cpu().numpy())

            next_obs, reward, done, _ = env.step(action)
            transitions.append((obs, action, reward))
            obs = next_obs
            step_count += 1

        return transitions

    def compute_returns(self, rewards: List[float]) -> torch.Tensor:
        """
        Compute discounted returns for the list of rewards.
        """
        R = 0.0
        returns = []
        for r in reversed(rewards):
            R = r + self.gamma * R
            returns.insert(0, R)
        return torch.tensor(returns, dtype=torch.float32, device=self.device)

    def inner_update(self, env: StopTargetTradingEnv, fmodel, diffopt):
        """
        Perform self.inner_steps gradient updates on the cloned model fmodel
        for the given environment.
        """
        for _ in range(self.inner_steps):
            transitions = self.collect_episodes(env, fmodel)
            if not transitions:
                continue

            obs_list, act_list, rew_list = zip(*transitions)
            obs_np = np.array(obs_list, dtype=np.float32)
            obs_t = torch.from_numpy(obs_np).to(self.device)
            returns_t = self.compute_returns(rew_list)

            logits = fmodel(obs_t)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e3, neginf=-1e3)
            log_probs = F.log_softmax(logits, dim=-1)

            chosen_log_probs = log_probs[range(len(act_list)), act_list]
            loss = - (returns_t * chosen_log_probs).mean()

            diffopt.step(loss)

        return fmodel

    def meta_update(self, envs: List[StopTargetTradingEnv]):
        """
        One meta-iteration over all tasks in envs:
         - Copy weights -> inner adaptation -> measure performance -> meta-loss
         - Then update the base policy with the mean of these losses.
        """
        meta_loss = 0.0
        task_count = len(envs)
        self.meta_optimizer.zero_grad()

        for env in envs:
            env.reset()
            inner_opt = torch.optim.SGD(self.policy.parameters(), lr=self.lr_inner)

            # Create a 'higher' context for MAML
            with higher.innerloop_ctx(self.policy, inner_opt, copy_initial_weights=True) as (fmodel, diffopt):
                # Inner adaptation
                self.inner_update(env, fmodel, diffopt)

                # gather transitions for meta-loss
                transitions = self.collect_episodes(env, fmodel)
                if not transitions:
                    continue

                obs_list, act_list, rew_list = zip(*transitions)
                obs_np = np.array(obs_list, dtype=np.float32)
                obs_t = torch.from_numpy(obs_np).to(self.device)
                returns_t = self.compute_returns(rew_list)

                # Evaluate the *base policy* (NOT the adapted fmodel) for meta-loss
                base_logits = self.policy(obs_t)
                base_logits = torch.nan_to_num(base_logits, nan=0.0, posinf=1e3, neginf=-1e3)
                base_log_probs = F.log_softmax(base_logits, dim=-1)
                chosen_log_probs = base_log_probs[range(len(act_list)), act_list]

                task_loss = - (returns_t * chosen_log_probs).mean()
                meta_loss += task_loss

        if task_count > 0:
            meta_loss = meta_loss / task_count
            meta_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.policy.parameters(), max_norm=self.grad_clip)
            self.meta_optimizer.step()

        return meta_loss.item() if task_count > 0 else 0.0

    def online_adaptation(self, new_env: StopTargetTradingEnv, adaptation_steps=1):
        """
        Adapt the current policy to new data using a few gradient steps
        directly on the existing policy (fine-tuning).
        """
        inner_opt = torch.optim.SGD(self.policy.parameters(), lr=self.lr_inner)

        for _ in range(adaptation_steps):
            transitions = self.collect_episodes(new_env, self.policy)
            if not transitions:
                continue

            obs_list, act_list, rew_list = zip(*transitions)
            obs_np = np.array(obs_list, dtype=np.float32)
            obs_t = torch.from_numpy(obs_np).to(self.device)
            returns_t = self.compute_returns(rew_list)

            logits = self.policy(obs_t)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e3, neginf=-1e3)
            log_probs = F.log_softmax(logits, dim=-1)
            chosen_log_probs = log_probs[range(len(act_list)), act_list]

            loss = - (returns_t * chosen_log_probs).mean()

            inner_opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.policy.parameters(), max_norm=self.grad_clip)
            inner_opt.step()


################################################################################
# 5. Train & Evaluate
################################################################################

def train_maml_on_multiple_tasks(df: pd.DataFrame, cfg: dict):
    """
    Create environment slices as tasks, run MAML for num_meta_iterations,
    and save final model state to cfg["model_file"].

    We no longer require 'StopLoss' or 'Target' columns, only 'close'.
    Stop and target are computed using fixed scale-invariant percentages
    (stop_pct, target_pct) specified in cfg.

    :param df: DataFrame with at least ['close', ...other numeric features...].
               No NaNs allowed.
    :param cfg: configuration dictionary
    :return: (policy, envs, maml_trader)
    """
    task_size = cfg["task_size"]
    num_tasks = cfg["num_tasks"]
    shuffle_tasks = cfg["shuffle_tasks"]
    num_meta_iterations = cfg["num_meta_iterations"]
    model_file = cfg["model_file"]
    seq_len = cfg["seq_len"]

    # environment hyperparams
    quantity = cfg["quantity"]
    holding_penalty = cfg["holding_penalty"]
    repeat_action_penalty = cfg["repeat_action_penalty"]
    reward_scale = cfg["reward_scale"]
    max_position_bars = cfg["max_position_bars"]
    max_bars_after_target = cfg["max_bars_after_target"]

    stop_pct = cfg["stop_pct"]
    target_pct = cfg["target_pct"]

    # Basic checks
    if len(df) < task_size:
        raise ValueError(f"DataFrame must have at least {task_size} rows, found {len(df)}.")
    if 'close' not in df.columns:
        raise ValueError("DataFrame must contain 'close' column.")

    # Check for NaNs
    if df.isna().any().any():
        raise ValueError("DataFrame has NaN values. Please clean them first.")

    num_features = df.shape[1]
    policy = DeeperTransformerPolicy(num_features, 4).to(device)

    # Try loading existing model weights
    if os.path.exists(model_file):
        print(f"Loading existing policy weights from '{model_file}'...")
        # "weights_only=True" is a PyTorch 2.0+ argument:
        #   torch.load(..., weights_only=True)
        # For broader compatibility, we'll just do the standard load:
        state_dict = torch.load(model_file, map_location=device)
        policy.load_state_dict(state_dict)
    else:
        print("No existing policy found. Creating a fresh policy...")

    # Instantiate the MAML trader
    maml_trader = AdvancedMAMLTrader(policy, cfg)

    # Prepare the task slices
    envs = []
    indices = []
    if shuffle_tasks:
        for _ in range(num_tasks):
            start = random.randint(0, len(df) - task_size)
            end = start + task_size
            indices.append((start, end))
    else:
        chunk_size = len(df) // num_tasks
        for i in range(num_tasks):
            st = i * chunk_size
            ed = st + task_size
            if ed > len(df):
                ed = len(df)
            indices.append((st, ed))

    for st, ed in indices:
        slice_df = df.iloc[st:ed].copy()
        env = StopTargetTradingEnv(
            df=slice_df,
            seq_len=seq_len,
            quantity=quantity,
            holding_penalty=holding_penalty,
            repeat_action_penalty=repeat_action_penalty,
            reward_scale=reward_scale,
            max_position_bars=max_position_bars,
            max_bars_after_target=max_bars_after_target,
            stop_pct=stop_pct,
            target_pct=target_pct
        )
        envs.append(env)

    # Meta-training loop
    for iteration in range(num_meta_iterations):
        meta_loss = maml_trader.meta_update(envs)
        print(f"Iteration {iteration+1}/{num_meta_iterations}, Meta-Loss: {meta_loss:.6f}")

    # Save trained model
    torch.save(maml_trader.policy.state_dict(), model_file)
    print(f"Policy weights saved to '{model_file}'")

    return maml_trader.policy, envs, maml_trader


def evaluate_policy(
    env: StopTargetTradingEnv,
    policy: nn.Module,
    initial_capital: float = 1_000_000.0,
    max_steps: int = 200
):
    """
    Evaluate the policy in GREEDY mode (argmax on action probabilities).
    Compute final capital based on compounding returns of each trade.

    :param env: instance of StopTargetTradingEnv
    :param policy: PyTorch policy
    :param initial_capital: float, starting capital for compounding
    :param max_steps: max steps to run in the environment
    :return: dict with TradeLog, AvgPnL%, WinRate, and FinalCapital
    """
    policy.eval()
    obs = env.reset()
    done = False
    step_count = 0

    while not done and step_count < max_steps:
        obs_np = np.array([obs], dtype=np.float32)
        obs_t = torch.from_numpy(obs_np).to(device)

        with torch.no_grad():
            logits = policy(obs_t)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e3, neginf=-1e3)
            # Greedy action = argmax on probabilities
            probs = F.softmax(logits, dim=-1)
            probs = torch.nan_to_num(probs, nan=1e-8)
            probs = probs / probs.sum(dim=-1, keepdim=True)
            action = probs[0].argmax().cpu().item()

        obs, _, done, _ = env.step(action)
        step_count += 1

    # Summarize results
    trades = env.get_trade_log()

    if len(trades) > 0:
        avg_pnl = trades['PnL%'].mean()
        wins = sum(trades['PnL%'] > 0)
        total = len(trades)
        win_rate = wins / total if total > 0 else 0.0

        # Compute final capital by compounding each trade's PnL%
        capital = initial_capital
        for _, row in trades.iterrows():
            capital *= (1.0 + (row['PnL%'] / 100.0))
    else:
        avg_pnl = 0.0
        win_rate = 0.0
        capital = initial_capital

    return {
        "TradeLog": trades,
        "AvgPnL%": avg_pnl,
        "WinRate": win_rate,
        "FinalCapital": capital
    }


################################################################################
# 6. Example Main Execution
################################################################################

if __name__ == "__main__":
    """
    Example usage:
      1) Ensure you have a dataframe 'processed_train_df' with columns:
         ['close', ...any other numeric features...].
      2) No NaNs or missing data.
      3) Then run this script/notebook cells in order.
      4) The environment will use stop_pct/target_pct from config
         to define the stops/targets. Rewards are scale-invariant.
    """
    try:
        print("Starting MAML training with scale-invariant Stop/Target environment...\n")

        # Provide your DataFrame externally, for example:
        # processed_train_df = pd.read_csv("your_data.csv")
        # processed_train_df.dropna(inplace=True)

        trained_policy, envs, maml_trader = train_maml_on_multiple_tasks(
            df=processed_train_df,
            cfg=config
        )

        print("\n--- MAML Training Complete ---\n")

        # Evaluate on the first environment slice (just as a test)
        test_env = envs[0]
        eval_res = evaluate_policy(
            env=test_env,
            policy=trained_policy,
            initial_capital=config["initial_capital"],
            max_steps=config["max_steps_per_env"]
        )
        print(f"Avg PnL%: {eval_res['AvgPnL%']:.4f}")
        print(f"Win Rate: {eval_res['WinRate']:.2%}")
        print(f"Final Capital: {eval_res['FinalCapital']:.2f}")
        print("Trade Log (head):\n", eval_res["TradeLog"].head(10))

    except NameError:
        print("ERROR: 'processed_train_df' not found. Please define your DataFrame before running.")
    except Exception as e:
        print("Exception:", str(e))

Using device: cuda
Starting MAML training with scale-invariant Stop/Target environment...

No existing policy found. Creating a fresh policy...
Iteration 1/50, Meta-Loss: -0.730795
Iteration 2/50, Meta-Loss: -0.763824
Iteration 3/50, Meta-Loss: -0.788844
Iteration 4/50, Meta-Loss: -0.776582
Iteration 5/50, Meta-Loss: -0.794505
Iteration 6/50, Meta-Loss: -0.820044
Iteration 7/50, Meta-Loss: -0.825410
Iteration 8/50, Meta-Loss: -0.778784
Iteration 9/50, Meta-Loss: -0.750234
Iteration 10/50, Meta-Loss: -0.766679
Iteration 11/50, Meta-Loss: -0.746390
Iteration 12/50, Meta-Loss: -0.785687
Iteration 13/50, Meta-Loss: -0.777020
Iteration 14/50, Meta-Loss: -0.772774
Iteration 15/50, Meta-Loss: -0.774459
Iteration 16/50, Meta-Loss: -0.746055
Iteration 17/50, Meta-Loss: -0.772397
Iteration 18/50, Meta-Loss: -0.775716
Iteration 19/50, Meta-Loss: -0.773967
Iteration 20/50, Meta-Loss: -0.760884
Iteration 21/50, Meta-Loss: -0.757683
Iteration 22/50, Meta-Loss: -0.756260
Iteration 23/50, Meta-Loss: -

In [13]:
print(f"Final Capital: {eval_res['FinalCapital']:.2f}")

Final Capital: 964419.67


In [14]:
eval_res["TradeLog"]

,Step,Position,EntryPrice,ExitPrice,PnL%,StopHit,TargetHit
0,61,short,29198.0,29497.5,-1.025755,True,False
1,66,short,29497.5,30252.2,-2.558522,True,False


In [15]:
def get_sleep_time(interval_minutes, market_start_hour=9, market_start_minute=15, market_close_hour=15, market_close_minute=0):
    # Get current time in IST
    now = datetime.now(pytz.utc).astimezone(ist_timezone)

    # Define the market start and close times in IST for today
    market_start_time = now.replace(hour=market_start_hour, minute=market_start_minute, second=0, microsecond=0)
    market_close_time = now.replace(hour=market_close_hour, minute=market_close_minute, second=0, microsecond=0)

    if now < market_start_time:
        # If current time is before market starts, set next_run_time to market start time
        next_run_time = market_start_time
    elif now > market_close_time:
        # If current time is after market close, calculate the time until the next market open
        next_market_start_time = market_start_time + timedelta(days=1)
        next_run_time = next_market_start_time
    else:
        # Calculate the minutes since the market start time
        minutes_since_market_start = (now - market_start_time).total_seconds() // 60
        # Calculate the number of minutes to the next interval boundary
        minutes_to_next_interval = interval_minutes - (minutes_since_market_start % interval_minutes)
        # Calculate the next run time by adding these minutes to the current time
        next_run_time = (now + timedelta(minutes=minutes_to_next_interval)).replace(second=0, microsecond=0)

    # Calculate the sleep time in seconds
    sleep_time = (next_run_time - now).total_seconds()
    return sleep_time

In [16]:
def fetch_option_chain():
    while True:
        try:
            data = {
                "symbol": index_symbol,
                "strikecount": 2,
                "timestamp": ""
            }
            response = fyers.optionchain(data=data)

            if response is not None:
                return response
        except Exception as e:
            print(f"Error fetching Option Chain: {e}")
            time.sleep(active_order_sleep)

index_oc= fetch_option_chain()

pd.DataFrame(index_oc['data']['optionsChain'])

,ask,bid,description,ex_symbol,exchange,fp,fpch,fpchp,fyToken,ltp,ltpch,ltpchp,option_type,strike_price,symbol,oi,oich,oichp,prev_oi,volume
0,0.00,0.00,NIFTYBANK-INDEX,BANKNIFTY,NSE,48671.0,-216.65,-0.44,101000000026009,48589.00,-135.40,-0.28,,-1,NSE:NIFTYBANK-INDEX,NaN,NaN,NaN,NaN,NaN
1,620.65,617.90,NaN,NaN,NaN,NaN,NaN,NaN,101125013039475,621.25,-189.95,-23.42,CE,48400,NSE:BANKNIFTY25JAN48400CE,305460.0,26385.0,9.45,279075.0,2019060.0
2,360.85,359.25,NaN,NaN,NaN,NaN,NaN,NaN,101125013039476,361.10,37.00,11.42,PE,48400,NSE:BANKNIFTY25JAN48400PE,278130.0,-23790.0,-7.88,301920.0,5294760.0
3,402.00,400.00,NaN,NaN,NaN,NaN,NaN,NaN,101125013039478,400.00,30.55,8.27,PE,48500,NSE:BANKNIFTY25JAN48500PE,1013685.0,29835.0,3.03,983850.0,13873800.0
4,563.30,562.20,NaN,NaN,NaN,NaN,NaN,NaN,101125013039477,563.60,-181.55,-24.36,CE,48500,NSE:BANKNIFTY25JAN48500CE,822525.0,95520.0,13.14,727005.0,9136500.0
5,447.35,444.60,NaN,NaN,NaN,NaN,NaN,NaN,101125013039480,445.30,42.40,10.52,PE,48600,NSE:BANKNIFTY25JAN48600PE,420060.0,47670.0,12.80,372390.0,12073965.0
6,506.60,505.70,NaN,NaN,NaN,NaN,NaN,NaN,101125013039479,506.05,-179.05,-26.13,CE,48600,NSE:BANKNIFTY25JAN48600CE,475365.0,106005.0,28.70,369360.0,10619865.0
7,496.30,493.90,NaN,NaN,NaN,NaN,NaN,NaN,101125013039482,493.40,44.90,10.01,PE,48700,NSE:BANKNIFTY25JAN48700PE,457830.0,154635.0,51.00,303195.0,12978105.0
8,456.60,453.45,NaN,NaN,NaN,NaN,NaN,NaN,101125013039481,456.55,-170.40,-27.18,CE,48700,NSE:BANKNIFTY25JAN48700CE,733035.0,283380.0,63.02,449655.0,13307430.0
9,548.10,545.00,NaN,NaN,NaN,NaN,NaN,NaN,101125013039485,547.90,56.15,11.42,PE,48800,NSE:BANKNIFTY25JAN48800PE,223095.0,-25425.0,-10.23,248520.0,9379500.0


In [17]:
def assign_ce_pe_option_symbols():
    symbol_oc = fetch_option_chain()

    if symbol_oc != None:
        # Convert the response data into a DataFrame
        oc_df = pd.DataFrame(symbol_oc['data']['optionsChain'])

        # Find the first 'CE' symbol from the top
        first_ce_symbol = None
        for index, row in oc_df.iterrows():
            if row['option_type'] == 'CE':
                first_ce_symbol = row['symbol']
                first_ce_strike = row['strike_price']
                break

        # Find the first 'PE' symbol from the bottom
        first_pe_symbol = None
        for index, row in oc_df[::-1].iterrows():  # Iterate in reverse
            if row['option_type'] == 'PE':
                first_pe_symbol = row['symbol']
                first_pe_strike = row['strike_price']
                break

        return first_ce_symbol, first_pe_symbol, first_ce_strike, first_pe_strike

In [18]:
def onmessage_ce(ce_message):
    global ce_ltp, index_ltp, unsubscribe_done
    try:
        if ce_message['symbol'] == ce_symbol:
            if "ltp" in ce_message:
                ce_ltp = ce_message["ltp"]
                ce_ltp = float(ce_ltp)

        elif ce_message['symbol'] == index_symbol:
            if "ltp" in ce_message:
                index_ltp = ce_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [ce_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {ce_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessageCE): {e}")


def onerror_ce(message):
    print("CE Error:", message)


def onclose_ce(message):
    print("CE Connection closed:", message)


def onopen_ce():

    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [ce_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def ce_buy_sell_ltp():
    global buy_sell_checked, ce_symbol, ce_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching CE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if ce_symbol is not None and ce_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                ce_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_ce,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_ce,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_ce,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_ce             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                ce_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [19]:
def onmessage_pe(pe_message):
    global pe_ltp, index_ltp, unsubscribe_done
    try:
        if pe_message['symbol'] == pe_symbol:
            if "ltp" in pe_message:
                pe_ltp = pe_message["ltp"]
                pe_ltp = float(pe_ltp)

        elif pe_message['symbol'] == index_symbol:
            if "ltp" in pe_message:
                index_ltp = pe_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [pe_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {pe_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessagePE): {e}")

def onerror_pe(message):
    print("PE Error:", message)


def onclose_pe(message):
    print("PE Connection closed:", message)


def onopen_pe():
    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [pe_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def pe_buy_sell_ltp():
    global buy_sell_checked, pe_symbol, pe_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching PE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if pe_symbol is not None and pe_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                pe_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_pe,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_pe,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_pe,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_pe             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                pe_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [20]:
def place_order(symbol):
    try:
        market_order_data = {
            "symbol": symbol,
            "qty": int(quantity),
            "type": 2,  # Market Order
            "side": 1,
            "productType": "INTRADAY",
            "limitPrice": 0,
            "stopPrice": 0,
            "validity": "DAY",
            "disclosedQty": 0,
            "offlineOrder":False
        }

        market_order_entry = fyers.place_order(data=market_order_data)

        if "id" in market_order_entry:
            market_order_id = market_order_entry["id"]
            market_order_message = market_order_entry["message"]
            print(f"{market_order_message}")

    except Exception as e:
        print(f"Error placing orders: {str(e)}")

In [21]:
def trail_order(symbol, stoploss):
    while True:
        try:
            stoploss = int(stoploss)
            pending_order = fyers.orderbook()

            matching_orders = [order for order in pending_order["orderBook"] if order["status"] == 6]

            modified_orders = 0

            for order in matching_orders:
                if order['symbol'] == symbol:
                    pending_order_id = order['id']
                    pending_order_side = order['side']
                    pending_order_side = int(pending_order_side)

                    if pending_order_side != 1:
                        data = {
                            "id": pending_order_id,
                            "type": 4,
                            "limitPrice": stoploss - 1,
                            "stopPrice": stoploss
                        }

                        modify = fyers.modify_order(data=data)
                        trail_message = modify["message"]
                        print(f"{trail_message}")

                        if trail_message == "Successfully modified order":
                            modified_orders += 1

            # Check if all matching orders are successfully modified
            if modified_orders == len(matching_orders):
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print("Error modifying order:" + str(e))

In [22]:
def exit_active_order(symbol):
    while True:
        try:
            data = {
                "id":f"{symbol}-INTRADAY"
            }

            exit_response = fyers.exit_positions(data=data)

            if ["message"] in exit_response:
                print(exit_response["message"])
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print(f"Error exiting Order: {e}")

In [23]:
def reset_flags():
    global active_order, buy_sell_checked

    active_order = False
    buy_sell_checked = False

In [24]:
# Function to save profits and losses
def save_overall(overall_win, overall_loss, capital):
    trade_type = {
        "overall_win": overall_win,
        "overall_loss": overall_loss,
        "capital": capital
    }

    with open("trade_data.json", "w") as file:
        json.dump(trade_type, file)


# Function to load wins and losses
def load_overall():
    try:
        with open('trade_data.json') as file:
            return json.load(file)
    except FileNotFoundError:
        return None

In [25]:
def handle_active_ce_order():
    def ce_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, ce_ltp, index_ltp, ce_strike, ce_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        ce_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if ce_ltp != 0 and index_ltp != 0:
                    ce_ltp_array.append(ce_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)


                    if index_ltp <= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(ce_symbol)

                        points = int(ce_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("CE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp >= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                            trailing_sl_inside = int(ce_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp - stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                    else:
                        if (index_ltp - prev_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(ce_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp - trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("CE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=ce_order_loop()).start()

In [26]:
def handle_active_pe_order():
    def pe_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, pe_ltp, index_ltp, pe_strike, pe_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        pe_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if pe_ltp != 0 and index_ltp != 0:
                    pe_ltp_array.append(pe_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)

                    if index_ltp >= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(pe_symbol)

                        points = int(pe_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("PE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp_array}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp <= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                            trailing_sl_inside = int(pe_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp + stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                    else:
                        if (prev_ltp - index_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(pe_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp + trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("PE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=pe_order_loop()).start()

In [27]:
def ce_entry():
    threading.Thread(target=ce_buy_sell_ltp).start()

    def ce_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if ce_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = ce_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp + target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp - trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(ce_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_ce_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=ce_entry_thread()).start()

In [28]:
def pe_entry():
    threading.Thread(target=pe_buy_sell_ltp).start()

    def pe_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if pe_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = pe_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp - target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp + trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(pe_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_pe_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=pe_entry_thread()).start()

In [29]:
def market_entry_exit_logic(action, actual_closing_price, final_df):
    global sl_hit_condition, unsubscribe_done, ce_ltp, pe_ltp, index_ltp, fixed_ltp, fixed_index_ltp, prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, ce_strike, pe_strike, ce_symbol, pe_symbol

    ce_ltp = 0
    pe_ltp  =0
    index_ltp = 0
    fixed_ltp = 0
    fixed_index_ltp = 0
    prev_ltp = 0
    target_inside = 0
    target_index_inside = 0
    trailing_sl_inside = 0
    trailing_index_inside = 0
    ce_strike = None
    pe_strike = None
    ce_symbol = None
    pe_symbol = None

    final_current_time = final_df.index[-1].time()
    print(final_current_time)

    # Ensure trading occurs only during market hours
    if final_current_time >= dt_time(9, (15 + interval_minutes)) and final_current_time <= dt_time(18, 0):
        if action == 1 and not active_order:  # Buy action
            print("Buy action detected by PPO model.")
            pygame.mixer.music.play()
            sl_hit_condition = False
            unsubscribe_done = False
            print(f"Making CE Position at {actual_closing_price}")
            ce_entry()  # Execute CE entry logic
        elif action == 2 and not active_order:  # Sell action
            print("Sell action detected by PPO model.")
            pygame.mixer.music.play()
            sl_hit_condition = False
            unsubscribe_done = False
            print(f"Making PE Position at {actual_closing_price}")
            pe_entry()  # Execute PE entry logic
        elif action == 0:
            print("Hold action detected by PPO model. No trade executed.")

In [30]:
def fetch_and_prepare_final_data():
    final_df = fetch_candle_data(10)

    final_data_processor = DataProcessor(final_df, live_processing=True)

    final_processed_df = final_data_processor.process().df

    return final_processed_df

In [31]:
def find_local_extrema(df):
    order=5
    atr_multiplier=1.5
    min_distance=5

    # Find local maxima and minima
    local_max = argrelextrema(df['high'].values, np.greater_equal, order=order)[0]
    local_min = argrelextrema(df['low'].values, np.less_equal, order=order)[0]

    # Calculate the threshold based on ATR
    threshold = df['atr'] * atr_multiplier

    # Filter by significance
    significant_max = []
    significant_min = []

    for idx in local_max:
        if idx > order and idx < len(df) - order:
            high = df['high'].iloc[idx]
            if significant_min:
                low = df['low'].iloc[significant_min[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_max.append(idx)
            else:
                significant_max.append(idx)

    for idx in local_min:
        if idx > order and idx < len(df) - order:
            low = df['low'].iloc[idx]
            if significant_max:
                high = df['high'].iloc[significant_max[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_min.append(idx)
            else:
                significant_min.append(idx)

    # Ensure minimum distance
    def filter_by_distance(points, min_distance):
        filtered_points = []
        for i in range(len(points)):
            if not filtered_points or (points[i] - filtered_points[-1]) > min_distance:
                filtered_points.append(points[i])
        return filtered_points

    significant_max = filter_by_distance(significant_max, min_distance)
    significant_min = filter_by_distance(significant_min, min_distance)

    return significant_max, significant_min

In [32]:
def get_trendline(df, point1, point2, kind='high'):
    x = [point1, point2]
    if kind == 'high':
        y = df['high'].values[x]
    else:
        y = df['low'].values[x]
    coeffs = np.polyfit(x, y, 1)
    trendline = np.polyval(coeffs, range(len(df)))
    return trendline

In [33]:
# Online training configuration
online_learning_steps = 1000  # Steps for each retraining
retrain_interval_minutes = 60  # Retrain the model every 60 minutes

start_time = datetime.now()

while True:
    clear_output(wait=True)

    num_candles = 100

    final_df = fetch_and_prepare_final_data()

    final_df = final_df.iloc[:-1]

    target = final_df['Target'].iloc[-1]
    trailing_sl = final_df['Stop Loss'].iloc[-1]

    # Initialize the TradingEnvironment with the latest data
    live_env = create_env(final_df, config)
    obs = live_env.reset()

    # Use PPO model to predict action for the latest data point
    action, _ = ppo_model.predict(obs, deterministic=True)

    # Extract the actual closing price
    actual_closing_price = final_df['close'].iloc[-1]

    # Retrain PPO model at specified intervals
    if (datetime.now() - start_time).seconds >= retrain_interval_minutes * 60:
        print("Retraining PPO model with online data...")
        train_env = TradingEnvironment(final_df, config)  # Create a new training environment
        ppo_model.set_env(train_env)  # Update the PPO model's environment
        ppo_model.learn(total_timesteps=online_learning_steps)  # Retrain the model
        ppo_model.save(model_save_path)  # Save the updated model
        start_time = datetime.now()

    # Identify most recent high and low points
    recent_highs, recent_lows = find_local_extrema(final_df)

    most_recent_high = recent_highs[-1] if len(recent_highs) > 1 else None
    most_recent_low = recent_lows[-1] if len(recent_lows) > 1 else None

    high_trendline = [np.nan] * len(final_df)
    low_trendline = [np.nan] * len(final_df)

    if most_recent_high is not None:
        previous_high = recent_highs[-2] if len(recent_highs) > 2 else most_recent_high
        high_trendline = get_trendline(final_df, previous_high, most_recent_high, kind='high')

    if most_recent_low is not None:
        previous_low = recent_lows[-2] if len(recent_lows) > 2 else most_recent_low
        low_trendline = get_trendline(final_df, previous_low, most_recent_low, kind='low')

    # Prepare candlestick data for mplfinance
    actual_candles = final_df[-num_candles:].copy()

    # Create a DataFrame for mplfinance
    mpf_df = actual_candles[['open', 'high', 'low', 'close']]

    # Create addplot elements for predicted prices and actual close prices
    ap = [
        mpf.make_addplot(final_df['close'][-num_candles:], color='none', panel=0, secondary_y=False, label=f"Actual Price: {final_df['close'].iloc[-1]}"),
        #mpf.make_addplot(y_pred_ensemble_final_plot, color=(0.95, 0.38, 0.25, 1), panel=0, secondary_y=False, label=f'Predicted Prices ({y_pred_ensemble_final_plot[-1]:.2f})')
    ]

    # Add trendlines to the plot
    if most_recent_high is not None:
        ap.append(mpf.make_addplot(high_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

    if most_recent_low is not None:
        ap.append(mpf.make_addplot(low_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

    fig, axlist = mpf.plot(mpf_df, type='candle', style='binancedark', volume=False, addplot=ap,
                        title=f'Chart', ylabel='Price',
                        figsize=(14, 7), returnfig=True)

    for ax in axlist:
        ax.grid(False)

    axlist[0].legend()
    plt.show()

    # Execute market entry/exit logic
    market_entry_exit_logic(action, actual_closing_price, final_df)

    sleep_time = get_sleep_time(interval_minutes)
    print(f"Sleeping for {sleep_time}")
    time.sleep(sleep_time)

NameError: name 'DataProcessor' is not defined